In [1]:
import json
import math
import os
import re

import numpy as np
import sentencepiece as spm
import torch
from transformers import AutoTokenizer

from custom_tokenizer import WhitespaceTokenizer
from train_lm import TrainConfig, build_model, evaluate

DATA_DIR = "data"
EVAL_BATCH_SIZE = 4  # only affects memory use, not the results
device = "cuda" if torch.cuda.is_available() else "cpu"

# Eval documents are disjoint from the documents the tokenizers were trained on
with open("plwiki_eval.txt", encoding="utf-8") as f:
    eval_docs = [d for d in f.read().split("\n\n") if d.strip()]
val_texts = [line for doc in eval_docs for line in doc.split("\n") if line.strip()]
n_chars = sum(len(t) for t in val_texts)
n_words = sum(len(t.split()) for t in val_texts)
print(f"{len(val_texts)} lines, {n_words:,} words, {n_chars:,} characters")

bpe = AutoTokenizer.from_pretrained("gpt2")
ws = WhitespaceTokenizer.load(os.path.join(DATA_DIR, "ws_tokenizer.json"))
sp = spm.SentencePieceProcessor(model_file=os.path.join(DATA_DIR, "sp.model"))

tokenizers = {
    "bpe": (bpe.encode, bpe.decode),
    "whitespace": (ws.encode, ws.decode),
    "sentencepiece": (sp.encode, sp.decode),
}


def load_model(name):
    run_dir = os.path.join("runs", name)
    with open(os.path.join(run_dir, "log.json")) as f:
        config = TrainConfig(**json.load(f)["config"])
    config.micro_batch_size = EVAL_BATCH_SIZE
    model = build_model(config).to(device)
    model.load_state_dict(torch.load(os.path.join(run_dir, "model.pt"), map_location=device))
    return model.eval(), config


trained = [n for n in tokenizers if os.path.exists(os.path.join("runs", n, "model.pt"))]
print("trained models:", trained)

21390 lines, 841,878 words, 6,193,473 characters
trained models: ['bpe', 'whitespace', 'sentencepiece']


In [2]:
# evaluate() returns nats per token over the evaluated blocks, so
# total loss = nats * n_tokens, and perplexity = exp(total loss / n_units)
results = {}
for name in trained:
    model, config = load_model(name)
    ids = np.load(os.path.join(DATA_DIR, f"ids_{name}_val.npy"))
    nats, _ = evaluate(model, ids, config, device, n_chars)
    results[name] = {
        "word_ppl": math.exp(nats * len(ids) / n_words),
        "char_ppl": math.exp(nats * len(ids) / n_chars),
    }
    del model
    torch.cuda.empty_cache()

print(f"{'tokenizer':<15}{'word ppl':>12}{'char ppl':>12}")
for name, r in results.items():
    print(f"{name:<15}{r['word_ppl']:>12.2f}{r['char_ppl']:>12.3f}")

tokenizer          word ppl    char ppl
bpe                34840.05       4.144
whitespace           510.17       2.334
sentencepiece       7335.68       3.353


In [3]:
# OOV words for the whitespace tokenizer: words only, punctuation tokens are not counted
words = [t for text in val_texts for t in ws.tokenize(text) if re.match(r"\w", t)]
oov = [t for t in words if t not in ws.token_to_id]
print(f"OOV word tokens: {len(oov):,} / {len(words):,} ({100 * len(oov) / len(words):.2f}%)")

types, oov_types = set(words), set(oov)
print(f"OOV word types: {len(oov_types):,} / {len(types):,} ({100 * len(oov_types) / len(types):.2f}%)")

OOV word tokens: 108,060 / 846,769 (12.76%)
OOV word types: 72,292 / 115,583 (62.55%)


In [4]:
@torch.no_grad()
def generate(name, prompt, n_new=40, temperature=0.8):
    encode, decode = tokenizers[name]
    model, config = load_model(name)
    ids = torch.tensor([encode(prompt)], device=device)
    for _ in range(n_new):
        logits = model(ids[:, -config.block_size :])[:, -1] / temperature
        next_id = torch.multinomial(torch.softmax(logits, dim=-1), 1)
        ids = torch.cat([ids, next_id], dim=1)
    return decode(ids[0].tolist())


torch.manual_seed(0)
for name in trained:
    print(name, "->", generate(name, "Warszawa jest"))

bpe -> Warszawa jest najpopularniejszych: "Ochot zachodzi" (dla występuje zagadnienie) oraz głos
whitespace -> warszawa jest siedzibą po bitwie na śląsku. w mieście znajduje się wzmianka o śląsku, w której znajduje się kilka osiedli, m. in.: miasto mazowieckie: <UNK>, <UNK>, <UNK>, <UNK>, <UNK>, <UNK>
sentencepiece -> Warszawa jest co najmniej 4, a 23 maja – 8 lutego – 22 lipca 2017, 12 października 2016 prezydent USA. Proponowane dwa z nich w historii kraju nielubowali na dziewięćdziesiątych lat. Ponadto w tym
